In [ ]:
!nvidia-smi

Mon Oct  5 14:05:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Часть 1. Подключение библиотек
`%%writefile` создаёт файл lab1.cu, следующие ячейки (`%%writefile -a`) дописывают в него код.

In [ ]:
%%writefile lab1.cu
// =====================================================================
// Лабораторная работа №1
// Точность и время работы функций sin, sinf, __sinf на GPU (float и double)
// =====================================================================

// ===== Часть 1. Подключение библиотек =====
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <cuda_runtime.h>

Overwriting lab1.cu


### Часть 2. Макрос проверки ошибок
Каждая CUDA-функция возвращает код ошибки. Макрос CHECK проверяет его и при ошибке выводит сообщение и завершает программу.

In [ ]:
%%writefile -a lab1.cu

// ===== Часть 2. Макрос проверки ошибок CUDA =====
#define CHECK(call)                                                   \
    do {                                                              \
        cudaError_t e = (call);                                       \
        if (e != cudaSuccess) {                                       \
            fprintf(stderr, "CUDA error: %s (%s:%d)\n",               \
                    cudaGetErrorString(e), __FILE__, __LINE__);       \
            exit(1);                                                  \
        }                                                             \
    } while (0)

Appending to lab1.cu


### Часть 3. Константы
N = 10^9 — размер массива; CHUNK — размер части для double; THREADS = 256 — потоков в блоке.
Pi задан отдельно для float и double, чтобы в float-ядре все вычисления шли в float.

In [ ]:
%%writefile -a lab1.cu

// ===== Часть 3. Константы =====
constexpr long long N     = 1000000000LL;  // размер массива: 10^9
constexpr long long CHUNK = 125000000LL;   // размер части для double (1 ГБ)
constexpr int THREADS     = 256;           // потоков в блоке

constexpr float  PI_F = 3.14159265358979f;         // Pi для float
constexpr double PI_D = 3.14159265358979323846;    // Pi для double

constexpr int F_SIN  = 0;   // функция sin
constexpr int F_SINF = 1;   // функция sinf
constexpr int F_INTR = 2;   // функция __sinf
const char* NAMES[] = {"sin   ", "sinf  ", "__sinf"};

Appending to lab1.cu


### Часть 4. Ядра для GPU
Каждый поток вычисляет один элемент массива: arr[i] = sin((i % 360) * Pi / 180).
Глобальный индекс потока: i = blockIdx.x * blockDim.x + threadIdx.x.
Параметр func выбирает функцию: sin, sinf или __sinf.
kernel_double обрабатывает часть массива double, начиная с индекса offset.

In [ ]:
%%writefile -a lab1.cu

// ===== Часть 4. Ядра (kernel) для GPU =====
__global__ void kernel_float(float* arr, long long n, int func) {
    long long i = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    float x = (i % 360) * PI_F / 180.0f;
    if      (func == F_SIN)  arr[i] = sin(x);
    else if (func == F_SINF) arr[i] = sinf(x);
    else if (func == F_INTR) arr[i] = __sinf(x);
}

__global__ void kernel_double(double* arr, long long offset, long long n, int func) {
    long long i = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    long long g = offset + i;
    double x = (g % 360) * PI_D / 180.0;
    if      (func == F_SIN)  arr[i] = sin(x);
    else if (func == F_SINF) arr[i] = sinf((float)x);
    else if (func == F_INTR) arr[i] = __sinf((float)x);
}

Appending to lab1.cu


### Часть 5. Подсчёт ошибки на CPU
REF — таблица точных значений sin для углов 0..359, вычисленная в double на CPU (эталон).
Сумма ошибок накапливается в double, так как при сложении 10^9 малых чисел во float теряется точность.

In [ ]:
%%writefile -a lab1.cu

// ===== Часть 5. Подсчёт ошибки на CPU =====
double REF[360];   // точные значения sin (эталон, double)

double error_float(const float* h, long long n) {
    double sum = 0.0;
    for (long long i = 0; i < n; ++i)
        sum += fabs(REF[i % 360] - (double)h[i]);
    return sum;
}

double error_double(const double* h, long long offset, long long n) {
    double sum = 0.0;
    for (long long i = 0; i < n; ++i)
        sum += fabs(REF[(offset + i) % 360] - h[i]);
    return sum;
}

Appending to lab1.cu


### Часть 6. Эксперимент с массивом float
Время ядра измеряется с помощью CUDA Events, ядро запускается 10 раз и время усредняется.
Затем массив копируется на CPU (cudaMemcpy) и считается средняя ошибка.

In [ ]:
%%writefile -a lab1.cu

// ===== Часть 6. Эксперимент с массивом float =====
constexpr int REPEATS = 10;   // число повторов ядра для усреднения времени

void test_float(float* d_arr, float* h_arr, int func, double* err, float* time_ms) {
    cudaEvent_t start, stop;
    CHECK(cudaEventCreate(&start));
    CHECK(cudaEventCreate(&stop));
    long long blocks = (N + THREADS - 1) / THREADS;

    // ядро запускается REPEATS раз, время усредняется
    CHECK(cudaEventRecord(start));
    for (int r = 0; r < REPEATS; ++r)
        kernel_float<<<(unsigned)blocks, THREADS>>>(d_arr, N, func);
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));
    CHECK(cudaGetLastError());
    float total_ms = 0.0f;
    CHECK(cudaEventElapsedTime(&total_ms, start, stop));
    *time_ms = total_ms / REPEATS;

    CHECK(cudaMemcpy(h_arr, d_arr, N * sizeof(float), cudaMemcpyDeviceToHost));
    *err = error_float(h_arr, N) / (double)N;

    CHECK(cudaEventDestroy(start));
    CHECK(cudaEventDestroy(stop));
}

Appending to lab1.cu


### Часть 7. Эксперимент с массивом double
Массив double из 10^9 элементов занимает 8 ГБ, поэтому обрабатывается частями по CHUNK элементов (8 частей по 1 ГБ).
Время всех частей суммируется, ошибка накапливается по всем частям.

In [ ]:
%%writefile -a lab1.cu

// ===== Часть 7. Эксперимент с массивом double (по частям) =====
void test_double(double* d_arr, double* h_arr, int func, double* err, float* time_ms) {
    cudaEvent_t start, stop;
    CHECK(cudaEventCreate(&start));
    CHECK(cudaEventCreate(&stop));
    double sum = 0.0;
    *time_ms = 0.0f;

    for (long long offset = 0; offset < N; offset += CHUNK) {
        long long n = (N - offset < CHUNK) ? (N - offset) : CHUNK;
        long long blocks = (n + THREADS - 1) / THREADS;
        float part_ms = 0.0f;

        // ядро запускается REPEATS раз, время усредняется
        CHECK(cudaEventRecord(start));
        for (int r = 0; r < REPEATS; ++r)
            kernel_double<<<(unsigned)blocks, THREADS>>>(d_arr, offset, n, func);
        CHECK(cudaEventRecord(stop));
        CHECK(cudaEventSynchronize(stop));
        CHECK(cudaGetLastError());
        CHECK(cudaEventElapsedTime(&part_ms, start, stop));
        *time_ms += part_ms / REPEATS;

        CHECK(cudaMemcpy(h_arr, d_arr, n * sizeof(double), cudaMemcpyDeviceToHost));
        sum += error_double(h_arr, offset, n);
    }
    *err = sum / (double)N;

    CHECK(cudaEventDestroy(start));
    CHECK(cudaEventDestroy(stop));
}

Appending to lab1.cu


### Часть 8. Главная функция
1. Заполняется таблица эталонных значений sin.
2. Выводится информация о GPU.
3. Выделяется память для float на GPU (cudaMalloc) и CPU (malloc).
4. Прогрев GPU — первый запуск ядра медленнее из-за инициализации.
5. Запускаются 6 экспериментов: 3 функции × 2 типа данных.
6. Освобождается память и выводится таблица результатов.

In [ ]:
%%writefile -a lab1.cu

// ===== Часть 8. Главная функция =====
int main() {
    for (int k = 0; k < 360; ++k)
        REF[k] = sin(k * PI_D / 180.0);

    cudaDeviceProp prop;
    CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s, память %.1f ГБ\n", prop.name, prop.totalGlobalMem / 1e9);
    printf("N = %lld\n\n", N);

    // float
    float* d_f = nullptr;
    CHECK(cudaMalloc(&d_f, N * sizeof(float)));
    float* h_f = (float*)malloc(N * sizeof(float));
    if (!h_f) { fprintf(stderr, "Ошибка malloc\n"); return 1; }

    kernel_float<<<(unsigned)((N + THREADS - 1) / THREADS), THREADS>>>(d_f, N, F_SINF); // прогрев
    CHECK(cudaDeviceSynchronize());

    double errF[3]; float timeF[3];
    for (int f = 0; f < 3; ++f)
        test_float(d_f, h_f, f, &errF[f], &timeF[f]);
    CHECK(cudaFree(d_f));
    free(h_f);

    // double
    double* d_d = nullptr;
    CHECK(cudaMalloc(&d_d, CHUNK * sizeof(double)));
    double* h_d = (double*)malloc(CHUNK * sizeof(double));
    if (!h_d) { fprintf(stderr, "Ошибка malloc\n"); return 1; }

    double errD[3]; float timeD[3];
    for (int f = 0; f < 3; ++f)
        test_double(d_d, h_d, f, &errD[f], &timeD[f]);
    CHECK(cudaFree(d_d));
    free(h_d);

    // результаты
    printf("Функция |  float: err       time, мс |  double: err      time, мс\n");
    printf("--------+--------------------------+---------------------------\n");
    for (int f = 0; f < 3; ++f)
        printf("%s  |  %.6e  %9.3f  |  %.6e  %9.3f\n",
               NAMES[f], errF[f], timeF[f], errD[f], timeD[f]);
    return 0;
}

Appending to lab1.cu


### Компиляция
-O2 — оптимизация, -arch=sm_75 — архитектура GPU Tesla T4 (Compute Capability 7.5), -o — имя исполняемого файла.
Флаг -use_fast_math не используется, иначе sinf будет заменена на __sinf.

In [ ]:
!nvcc -O2 -arch=sm_75 lab1.cu -o lab1_exec

### Запуск программы
Время в таблице — среднее по 10 запускам ядра.

In [ ]:
!./lab1_exec

GPU: Tesla T4, память 15.6 ГБ
N = 1000000000

Функция |  float: err       time, мс |  double: err      time, мс
--------+--------------------------+---------------------------
sin     |  7.599349e-08     23.369  |  8.779628e-18    260.434
sinf    |  7.599349e-08     30.326  |  4.679493e-08    115.515
__sinf  |  1.148449e-07     27.150  |  1.301495e-07    115.367
